# Transition tables for Cifar100

In [5]:
import pandas as pd
import re
from pathlib import Path
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path

In [6]:
def fix_results_csv(in_path: str, out_path: str) -> str:
    """
    Wrap lb_minus_rhs (which looks like [[inf], [inf], ...]) in quotes so commas inside
    don't break CSV parsing.
    Assumes the file columns are:
      instance_id,onnx,vnnlib,timeout,result,lb_minus_rhs,domains_visited,bab_time,all_time,init_unstable
    """
    with open(in_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.read().splitlines()

    if not lines:
        raise ValueError("Empty file")

    fixed = [lines[0]]  # header
    for line in lines[1:]:
        if not line.strip():
            continue

        # Only repair lines that contain the list field in unquoted form.
        # We detect the start of lb_minus_rhs by looking for ',[[' after the 'result' field.
        if ",[[" not in line:
            fixed.append(line)
            continue

        prefix, rest = line.split(",[[", 1)
        lb_and_after = "[[" + rest

        end_idx = lb_and_after.find("]]")
        if end_idx == -1:
            # Malformed; keep original so you can spot it later
            fixed.append(line)
            continue

        lb_str = lb_and_after[: end_idx + 2]   # includes final ']]'
        suffix = lb_and_after[end_idx + 2 :]   # starts with ,domains_visited,...

        # Quote the whole field; double any quotes inside (CSV escaping)
        lb_escaped = lb_str.replace('"', '""')
        fixed.append(prefix + ',"' + lb_escaped + '"' + suffix)

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(fixed) + "\n")

    return out_path

In [7]:
experiment = "cifar100_segmented_d_1"

results_path = f"../results/{experiment}/results.csv"
results_fixed = f"../results/{experiment}/results.fixed.csv"
stats_path = f"../results/{experiment}/input_change_stats.csv"
out_path = f"../results/{experiment}/combined_results.csv"

fix_results_csv(results_path, results_fixed)
df_res = pd.read_csv(results_fixed)
df_ics = pd.read_csv(stats_path)

FileNotFoundError: [Errno 2] No such file or directory: '../results/cifar100_segmented_d_1/input_change_stats.csv'

In [ ]:
num_rx = re.compile(r"(-?inf|[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")

def parse_lb_list(s):
    if pd.isna(s):
        return None
    toks = num_rx.findall(str(s))
    vals = []
    for t in toks:
        t = t.lower()
        if t in ("inf", "+inf"):
            vals.append(math.inf)
        elif t == "-inf":
            vals.append(-math.inf)
        else:
            vals.append(float(t))
    return vals

def lb_summary(vals):
    if vals is None:
        return pd.Series({
            "lb_len": 0,
            "lb_num_finite": 0,
            "lb_min_finite": np.nan,
            "lb_argmin_finite": np.nan,
            "lb_any_neg_finite": False,
        })

    finite = [v for v in vals if math.isfinite(v)]
    if finite:
        minv = min(finite)
        argmin = vals.index(minv)
        any_neg = any(v < 0 for v in finite)
    else:
        minv = np.nan
        argmin = np.nan
        any_neg = False

    return pd.Series({
        "lb_len": len(vals),
        "lb_num_finite": len(finite),
        "lb_min_finite": minv,
        "lb_argmin_finite": argmin,
        "lb_any_neg_finite": any_neg,
    })

lb_vals = df_res["lb_minus_rhs"].apply(parse_lb_list)
lb_cols = lb_vals.apply(lb_summary)

df_res = pd.concat([df_res, lb_cols], axis=1)

# Optional: drop the messy original column entirely
df_res = df_res.drop(columns=["lb_minus_rhs"])


In [ ]:
# Accept BOTH:
# 1) image_global_kK_eps_E.vnnlib
# 2) image_seg0_fixmask_kK_eps_E.vnnlib
# 3) image_fix_mask_seg0_kK_eps_E.vnnlib
# plus fixnonmask / fix_nonmask variants
RX_GLOBAL = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_global_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

RX_SEG_A = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_seg(?P<seg>\d+)_(?P<tag>fixmask|fixnonmask|fix_mask|fix_nonmask)
_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

RX_SEG_B = re.compile(r"""(?x)
(?:.*/)?(?P<image>.+?)_(?P<tag>fixmask|fixnonmask|fix_mask|fix_nonmask)_seg(?P<seg>\d+)
_k(?P<k>\d+)_eps_(?P<eps>[0-9]*\.?[0-9]+)\.vnnlib$
""")

def normalize_tag(tag: str) -> str:
    tag = tag.lower()
    tag = tag.replace("fixmask", "fix_mask").replace("fixnonmask", "fix_nonmask")
    return tag

def parse_vnnlib(vnnlib_path: str):
    s = str(vnnlib_path)

    m = RX_GLOBAL.search(s)
    if m:
        return pd.Series({
            "image": m.group("image"),
            "tag": "global",
            "segment_index": -1,
            "k": int(m.group("k")),
            "eps": float(m.group("eps")),
        })

    for rx in (RX_SEG_A, RX_SEG_B):
        m = rx.search(s)
        if m:
            return pd.Series({
                "image": m.group("image"),
                "tag": normalize_tag(m.group("tag")),
                "segment_index": int(m.group("seg")),
                "k": int(m.group("k")),
                "eps": float(m.group("eps")),
            })

    # If something doesn't match, keep it visible (so you can debug)
    return pd.Series({"image": None, "tag": None, "segment_index": None, "k": None, "eps": None})

def strip_model_prefix(row):
    img = row["image"]
    model = row["model"]
    if isinstance(img, str) and isinstance(model, str):
        pref = model + "__"
        if img.startswith(pref):
            img = img[len(pref):]
    return img


# Parse keys from results.csv
parsed = df_res["vnnlib"].apply(parse_vnnlib)
df_res2 = pd.concat([df_res, parsed], axis=1)

# Extract model name from onnx path (onnx/vgg16-7.onnx -> vgg16-7)
df_res2["model"] = df_res2["onnx"].astype(str).apply(lambda p: Path(p).stem)

# Ensure types match (important for merging!)
df_ics2 = df_ics.copy()
df_ics2["image"] = df_ics2.apply(strip_model_prefix, axis=1)
df_ics2["k"] = df_ics2["k"].astype("int64")
df_ics2["segment_index"] = df_ics2["segment_index"].astype("int64")
df_ics2["eps"] = df_ics2["eps"].astype("float64")
df_ics2["model"] = df_ics2["model"].astype(str)

# Merge on the actual shared keys:
keys = ["image", "model", "eps", "k", "tag", "segment_index"]

merged = df_res2.merge(df_ics2, on=keys, how="left", suffixes=("_res", "_ics"))

# Sanity checks
matched = merged["pattern"].notna().sum()  # 'pattern' exists in input_change_stats.csv
total = len(merged)
print(f"Matched rows: {matched}/{total}  ({matched/total:.3f})")

# Show any rows that failed to parse or match
bad_parse = merged[merged["image"].isna()][["vnnlib"]].head(20)
if len(bad_parse):
    print("\nExamples that FAILED TO PARSE (fix regex for these):")
    print(bad_parse.to_string(index=False))

bad_match = merged[merged["pattern"].isna()][["vnnlib"] + keys].head(20)
if len(bad_match):
    print("\nExamples that PARSED but DIDN'T MATCH stats keys:")
    print(bad_match.to_string(index=False))

merged.to_csv(out_path, index=False)
print("Wrote:", out_path)

Matched rows: 600/600  (1.000)
Wrote: ../results/cifar100_segmented/combined_results.csv


-------

In [ ]:
import pandas as pd
df=pd.read_csv(out_path)

In [ ]:
def tag_to_semantic(t):
    t=str(t).lower()
    if "global" in t: return "Global"
    if "fix_mask" in t : return "Background"
    if "fix_nonmask" in t: return "Object"
    return "Unknown"
df["semantic"]=df["tag"].apply(tag_to_semantic)

In [ ]:
big=df.groupby(['k','eps','semantic']).size().unstack(fill_value=0)
big

,semantic,Background,Global,Object
k,eps,,,
1024,0.0039,200,200,200


In [ ]:
import numpy as np
import pandas as pd

bab_used = df["domains_visited"].fillna(0).astype(float) > 0
res = df["result"].astype(str).str.lower().str.strip()

margin = df["lb_min_finite"]  # <-- this replaces lb_minus_rhs
margin_is_nan = margin.isna()
no_finite = df["lb_num_finite"].fillna(0).astype(int) == 0

def compute_status(i):
    r = res.iat[i]
    bab = bool(bab_used.iat[i])

    if r.startswith("unsat"):
        return "Safe with branch-and-bound" if bab else "Safe via initial bounding"

    if r.startswith("sat"):
        return "Counterexample found during branch-and-bound" if bab else "Counterexample found during initial bounding/attack"

    if ("timeout" in r) or ("unknown" in r):
        return "Unknown with branch-and-bound" if bab else "Unknown without branch-and-bound"

    return "Other/Unparsed"

df["status"] = [compute_status(i) for i in range(len(df))]

In [ ]:
# ---- Short labels for paper
STATUS_SHORT = {
    "Counterexample found immediately": "CE",
    "Counterexample found during initial bounding/attack": "CE (init)",
    "Safe via initial bounding": "Safe (init)",
    "Safe with branch-and-bound": "Safe (B&B)",
    "Unknown with branch-and-bound": "Unk (B&B)",
    "Unknown without branch-and-bound": "Unk",
    "safe with branch-and-bound": "Safe (B&B)",  # in case of capitalization differences
    "Other/Unparsed": "Other",
}

# Apply mapping (keep original too)
df["status_short"] = df["status"].map(STATUS_SHORT).fillna(df["status"])

In [ ]:
import pandas as pd
import numpy as np


FULL_K = 1024  # total pixels in image

def pretty_k_label(k):
    """
    Convert k into fraction-of-image label.
    Examples:
        50176 -> "1"
        25088 -> "1/2"
        12544 -> "1/4"
        ...
    """
    ratio = FULL_K / k

    # Exact powers of two
    if ratio.is_integer():
        ratio = int(ratio)
        if ratio == 1:
            return "1"
        return f"1/{ratio}"

    # fallback: percentage
    pct = 100 * k / FULL_K
    return f"{pct:.1f}%"

ORDER = ["CE", "CE (init)", "Safe (init)", "Safe (B&B)", "Unk (B&B)", "Unk"]

def transition_crosstab(df, a="Global", b="Object", order=ORDER):
    pivot = df.pivot_table(
        index=["image", "model", "k", "eps"],
        columns="semantic",
        values="status_short",
        aggfunc="first"
    )

    t = pivot[[a, b]].dropna()
    tab = pd.crosstab(t[a], t[b], dropna=False)

    rows = list(order) + [x for x in tab.index if x not in order]
    cols = list(order) + [x for x in tab.columns if x not in order]

    tab = tab.reindex(index=rows, columns=cols, fill_value=0)

    # add totals
    tab.loc["TOTAL"] = tab.sum(axis=0)
    tab["TOTAL"] = tab.sum(axis=1)
    return tab

def style_transition_table(tab: pd.DataFrame, caption: str = ""):
    idx = tab.index.tolist()
    cols = tab.columns.tolist()

    # diagonal mask for matching labels (excluding TOTAL row/col)
    diag = pd.DataFrame(False, index=idx, columns=cols)
    for r in idx:
        if r == "TOTAL": 
            continue
        if r in cols:
            diag.loc[r, r] = True
    # exclude TOTAL column from diag highlight
    if "TOTAL" in cols:
        diag["TOTAL"] = False
    if "TOTAL" in idx:
        diag.loc["TOTAL", :] = False

    styler = (
        tab.style
        .set_caption(caption)
        .format("{:d}")
        .set_properties(**{
            "text-align": "center",
            "border": "1px solid #444",
            "padding": "6px",
            "font-size": "12px",
        })
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("text-align", "left")]},
            {"selector": "th", "props": [("background-color", "#f2f2f2"), ("font-weight", "bold")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]}
        ])
        # diagonal highlight
        .apply(lambda _: diag.map(lambda x: "background-color: #fff6b3; font-weight: bold;" if x else ""), axis=None)
        # emphasize totals
        .apply(lambda s: ["font-weight: bold; background-color: #eaeaea;" if s.name == "TOTAL" else "" for _ in s], axis=1)
        .apply(lambda s: ["font-weight: bold; background-color: #eaeaea;" if c == "TOTAL" else "" for c in s.index], axis=0)
    )
    return styler

def style_transition_table(tab: pd.DataFrame, caption: str = "", diag_color="#fff6b3", offdiag_color="#dbeafe"):
    """
    tab: crosstab WITH TOTAL row/col already added.
    diag_color: diagonal highlight (same->same)
    offdiag_color: any NON-ZERO off-diagonal highlight
    """

    idx = tab.index.tolist()
    cols = tab.columns.tolist()

    def cell_style(r, c, v):
        # base: force readability in dark themes too
        base = "background-color: white; color: #111827;"

        # totals row/col
        if r == "TOTAL" or c == "TOTAL":
            return "background-color: #f1f5f9; color: #111827; font-weight: 700;"

        # diagonal (same label), highlight
        if r == c:
            # yellow + bold + black numbers
            return f"background-color: {diag_color}; color: #111827; font-weight: 800;"

        # off-diagonal non-zero highlight
        if v != 0:
            return f"background-color: {offdiag_color}; color: #111827; font-weight: 800;"

        # zero cells: keep clean
        return base

    # Build a full style DataFrame
    styles = pd.DataFrame("", index=idx, columns=cols)
    for r in idx:
        for c in cols:
            styles.loc[r, c] = cell_style(r, c, int(tab.loc[r, c]))

    styler = (
        tab.style
        .set_caption(caption)
        .format("{:d}")
        # apply our per-cell styles
        .apply(lambda _: styles, axis=None)
        # table-level CSS (forces a nice paper look)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("font-weight", "800"),
                ("text-align", "left"),
                ("color", "#111827"),
                ("font-size", "13px"),
                ("padding", "6px 0"),
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"),
                ("background-color", "white"),
            ]},
            {"selector": "th", "props": [
                ("background-color", "white"),
                ("color", "#111827"),
                ("font-weight", "800"),
                ("border", "1px solid #cbd5e1"),
                ("padding", "6px 8px"),
            ]},
            {"selector": "td", "props": [
                ("border", "1px solid #cbd5e1"),
                ("padding", "6px 8px"),
                ("text-align", "center"),
            ]},
        ])
    )

    return styler

def display_with_corner_label(styler, row_label="Global status", col_label="Object status"):
    # remove the extra index/column name header rows
    styler = styler.set_table_styles([
        {"selector": "th.row_heading", "props": [("white-space", "nowrap")]},
    ], overwrite=False)

    # hide index name + columns name (prevents the extra "Global status" row)
    styler = styler.set_table_styles([], overwrite=False)
    styler = styler.set_caption(styler.caption)

    # This is the key: blank out names so pandas doesn't create that extra header row
    styler.data.index.name = None
    styler.data.columns.name = None

    # Add a combined corner label (shown as first column header area in HTML via CSS)
    corner = f"{row_label} / {col_label}"
    return styler.set_table_styles([
        # put text into the top-left corner header cell
        {"selector": "th.blank.level0", "props": [
            ("background-color", "white"),
            ("color", "#111827"),
            ("font-weight", "800"),
            ("border", "1px solid #cbd5e1"),
        ]},
    ], overwrite=False).set_properties(subset=pd.IndexSlice[:, :], **{}), corner



In [ ]:
# Global -> Object
tab_GO = transition_crosstab(df, "Global", "Object")
tab_GO.index.name = "Global status"
tab_GO.columns.name = "Object status"
display(style_transition_table(tab_GO, "Global → Object (status transitions)"))

# Global -> Background
tab_GB = transition_crosstab(df, "Global", "Background")
tab_GB.index.name = "Global status"
tab_GB.columns.name = "Background status"
display(style_transition_table(tab_GB, "Global → Background (status transitions)"))

Object status,CE,CE (init),Safe (init),Safe (B&B),Unk (B&B),Unk,TOTAL
Global status,,,,,,,
CE,0,0,0,0,0,0,0
CE (init),0,18,3,0,0,4,25
Safe (init),0,0,103,0,0,0,103
Safe (B&B),0,0,0,0,0,0,0
Unk (B&B),0,0,0,0,0,0,0
Unk,0,0,45,0,0,27,72
TOTAL,0,18,151,0,0,31,200


Background status,CE,CE (init),Safe (init),Safe (B&B),Unk (B&B),Unk,TOTAL
Global status,,,,,,,
CE,0,0,0,0,0,0,0
CE (init),0,0,25,0,0,0,25
Safe (init),0,0,103,0,0,0,103
Safe (B&B),0,0,0,0,0,0,0
Unk (B&B),0,0,0,0,0,0,0
Unk,0,0,72,0,0,0,72
TOTAL,0,0,200,0,0,0,200


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------
# Build tables (your code)
# -------------------------------
tab_GO = transition_crosstab(df, "Global", "Object")
tab_GO.index.name = "Global status"
tab_GO.columns.name = "Object status"

tab_GB = transition_crosstab(df, "Global", "Background")
tab_GB.index.name = "Global status"
tab_GB.columns.name = "Background status"

ALL_STATUSES = ["CE", "CE (init)", "Safe (init)", "Safe (B&B)", "Unk (B&B)", "Unk", "TOTAL"]
tab_GO = tab_GO.reindex(index=ALL_STATUSES, columns=ALL_STATUSES, fill_value=0)
tab_GB = tab_GB.reindex(index=ALL_STATUSES, columns=ALL_STATUSES, fill_value=0)

# -------------------------------
# Pretty, compact table renderer
# -------------------------------
def draw_df_table_pretty(ax, df_table: pd.DataFrame, title: str, fontsize=9, title_fontsize=11):
    ax.axis("off")
    ax.set_title(title, fontsize=title_fontsize, pad=2)

    dfv = df_table.copy()
    cell_text = dfv.astype(int).astype(str).values.tolist()

    tbl = ax.table(
        cellText=cell_text,
        rowLabels=dfv.index.tolist(),
        colLabels=dfv.columns.tolist(),
        loc="center",
        cellLoc="center",
        colLoc="center",
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)

    # Compact: wider, not taller
    tbl.scale(1.18, 1.05)

    # Colors similar to your screenshot
    header_bg  = "#E9EEF6"   # header band
    rowlabel_bg= "#F2F2F2"   # row label band
    total_bg   = "#EFEFEF"   # totals shading
    diag_bg    = "#FFF3B0"   # diagonal highlight
    nonzero_bg = "#DCEBFF"   # nonzero highlight
    edge_col   = "#B8C4D6"

    # Base styling + tighter row height
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.6)
        cell.set_edgecolor(edge_col)
        cell.set_facecolor("white")
        cell.set_height(cell.get_height() * 0.92)

        # column header row
        if r == 0 and c >= 0:
            cell.set_facecolor(header_bg)
            cell.set_text_props(weight="bold")
            cell.set_linewidth(0.9)

        # row label column
        if c == -1 and r >= 1:
            cell.set_facecolor(rowlabel_bg)
            cell.set_text_props(weight="bold")
            cell.set_linewidth(0.9)

        # top-left corner
        if r == 0 and c == -1:
            cell.set_facecolor("#D6DCE8")
            cell.set_text_props(weight="bold")

    # Body semantic coloring
    rows = dfv.index.tolist()
    cols = dfv.columns.tolist()
    for i, rname in enumerate(rows):
        for j, cname in enumerate(cols):
            r = i + 1   # data rows start at 1 in mpl-table
            c = j       # data cols start at 0
            val = int(dfv.iloc[i, j])

            # totals row/col
            if rname == "TOTAL" or cname == "TOTAL":
                tbl[(r, c)].set_facecolor(total_bg)
                tbl[(r, c)].set_text_props(weight="bold")
                continue

            # diagonal
            if rname == cname:
                tbl[(r, c)].set_facecolor(diag_bg)
                if val != 0:
                    tbl[(r, c)].set_text_props(weight="bold")
                continue

            # nonzero
            if val != 0:
                tbl[(r, c)].set_facecolor(nonzero_bg)
                tbl[(r, c)].set_text_props(weight="bold")

    return tbl

# -------------------------------
# Side-by-side + tight PDF
# -------------------------------
out_pdf = Path("pdfs/transition_tables_side_by_side_d_1.pdf")

plt.rcParams.update({"font.family": "DejaVu Sans"})
fig, axes = plt.subplots(1, 2, figsize=(12.5, 2.2), dpi=250)

draw_df_table_pretty(axes[0], tab_GO, "Global → Object (status transitions)")
draw_df_table_pretty(axes[1], tab_GB, "Global → Background (status transitions)")

# Aggressive whitespace trimming
fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.02, wspace=0.40)

fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)

print("Saved:", out_pdf)

NameError: name 'transition_crosstab' is not defined